# ⚠️ PlasticSense AI — Severity Assessment Engine (Notebook 09)

### 🌟 Overview
This notebook builds a **complete Severity Assessment Engine** that converts YOLO detection outputs into an **environmental pollution severity score**. It combines object count, plastic density, hazard weights, and waterbody proximity to classify pollution into **Low / Medium / High / Critical** levels.

### 📥 Inputs
| Asset | Source |
|---|---|
| Trained Model | `PlasticSense_AI/models/best.pt` |
| Inference Functions | Reused from Notebook 08 |
| Image + GPS | User-provided (latitude, longitude) |

### 📤 Outputs
All severity results are saved under `PlasticSense_AI/severity/`:
```
severity/
├── json/          # Per-image severity JSON reports
├── csv/           # Tabular severity results
├── plots/         # Severity visualizations
└── reports/       # Batch summary reports
```

### 🧮 Severity Formula
```
severity_score = 0.35 × Normalized Object Count
               + 0.30 × Density Score
               + 0.25 × Hazard Score
               + 0.10 × Waterbody Score
```

### 🚦 Classification
| Score Range | Level | Badge |
|---|---|---|
| 0–25 | Low | 🟢 |
| 26–50 | Medium | 🟡 |
| 51–75 | High | 🟠 |
| 76–100 | Critical | 🔴 |

### ⚠️ Prerequisites
Run notebooks **01–08** first. This notebook does **NOT** retrain the model.

---
## 1. Environment Setup & Library Installation
Install and import all required dependencies for the severity engine.

In [ ]:
!pip install -q ultralytics rich pyyaml pandas opencv-python matplotlib pillow tqdm

In [ ]:
# ──────────────────────────────────────────────────────────
# Standard Library
# ──────────────────────────────────────────────────────────
import os
import sys
import json
import math
import time
import shutil
import logging
import datetime
import warnings
import textwrap
from pathlib import Path
from typing import Dict, List, Tuple, Any, Optional, Union
from collections import Counter

# ──────────────────────────────────────────────────────────
# Third-Party
# ──────────────────────────────────────────────────────────
import torch
import numpy as np
import pandas as pd
import cv2
import matplotlib
matplotlib.use('Agg')  # Non-interactive backend for saving
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image
from tqdm.auto import tqdm

from rich.console import Console
from rich.table import Table
from rich.panel import Panel
from rich.progress import track

from ultralytics import YOLO

# ──────────────────────────────────────────────────────────
# Configuration
# ──────────────────────────────────────────────────────────
warnings.filterwarnings('ignore')
plt.rcParams.update({
    'figure.dpi': 150,
    'savefig.dpi': 150,
    'font.size': 11,
    'axes.titlesize': 14,
    'axes.labelsize': 12,
    'figure.figsize': (12, 8),
    'savefig.bbox': 'tight',
    'savefig.pad_inches': 0.1
})

console = Console()
SEED = 42
np.random.seed(SEED)

console.print('[bold green]✔ All libraries imported successfully.[/bold green]')

---
## 2. Logging System Initialization
Set up a dual-output logger (console + file) consistent with prior notebooks.

In [ ]:
def setup_logger(log_dir: Path, name: str = 'PlasticSense_Severity') -> logging.Logger:
    """Create a logger with file and console handlers."""
    log_dir.mkdir(parents=True, exist_ok=True)
    logger = logging.getLogger(name)
    logger.setLevel(logging.INFO)
    logger.handlers = []  # Reset

    fmt = logging.Formatter(
        '[%(asctime)s] %(levelname)s — %(message)s',
        datefmt='%Y-%m-%d %H:%M:%S'
    )

    # File handler
    fh = logging.FileHandler(log_dir / 'severity_engine.log', mode='w')
    fh.setFormatter(fmt)
    logger.addHandler(fh)

    # Console handler
    ch = logging.StreamHandler(sys.stdout)
    ch.setFormatter(fmt)
    logger.addHandler(ch)

    return logger

---
## 3. Hardware Verification
Detect available compute accelerator (CUDA / MPS / CPU).

In [ ]:
def check_hardware() -> str:
    """Detect hardware accelerator and display status table."""
    table = Table(title='Hardware & Environment Status', show_header=True)
    table.add_column('Component', style='cyan')
    table.add_column('Status / Version', justify='right')

    table.add_row('Python Version', sys.version.split()[0])
    table.add_row('PyTorch Version', torch.__version__)

    device_type = 'cpu'
    if torch.cuda.is_available():
        device_name = torch.cuda.get_device_name(0)
        table.add_row('GPU Accelerator', f'[green]✔ {device_name}[/green]')
        table.add_row('CUDA Version', str(torch.version.cuda))
        device_type = 'cuda:0'
    elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
        table.add_row('GPU Accelerator', '[green]✔ Apple Silicon (MPS)[/green]')
        device_type = 'mps'
    else:
        table.add_row('GPU Accelerator', '[bold red]✖ CPU ONLY[/bold red]')
        console.print('[bold yellow]⚠ GPU unavailable — inference will use CPU.[/bold yellow]')

    import ultralytics
    table.add_row('Ultralytics Version', ultralytics.__version__)

    console.print(table)
    return device_type

DEVICE = check_hardware()

---
## 4. Project Paths & Directory Structure
Define all paths and create the severity output directory tree.

In [ ]:
# ──────────────────────────────────────────────────────────
# Detect Environment: Colab vs Local
# ──────────────────────────────────────────────────────────
IS_COLAB = 'google.colab' in sys.modules

if IS_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = Path('/content/drive/MyDrive/PlasticSense_AI')
else:
    PROJECT_ROOT = Path('/Users/siddhivinayak/project/PlasticSense AI/Ml-model')

# ──────────────────────────────────────────────────────────
# Input Paths
# ──────────────────────────────────────────────────────────
MODELS_DIR     = PROJECT_ROOT / 'models'
BEST_PT_PATH   = MODELS_DIR / 'best.pt'
DATASET_DIR    = PROJECT_ROOT / 'Datasets' / 'augmented_yolo'
TEST_IMAGES    = DATASET_DIR / 'images' / 'test'

# ──────────────────────────────────────────────────────────
# Output Paths — Severity Engine
# ──────────────────────────────────────────────────────────
SEVERITY_DIR    = PROJECT_ROOT / 'severity'
SEV_JSON_DIR    = SEVERITY_DIR / 'json'
SEV_CSV_DIR     = SEVERITY_DIR / 'csv'
SEV_PLOTS_DIR   = SEVERITY_DIR / 'plots'
SEV_REPORTS_DIR = SEVERITY_DIR / 'reports'
SEV_LOGS_DIR    = SEVERITY_DIR / 'logs'

# Create all output directories
for d in [SEVERITY_DIR, SEV_JSON_DIR, SEV_CSV_DIR,
          SEV_PLOTS_DIR, SEV_REPORTS_DIR, SEV_LOGS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# Initialize logger
logger = setup_logger(SEV_LOGS_DIR)
logger.info('Severity Engine pipeline initialized.')

# Supported image extensions (reused from Notebook 08)
SUPPORTED_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.webp'}

console.print(Panel.fit(
    f'[bold cyan]Project Root:[/bold cyan]     {PROJECT_ROOT}\n'
    f'[bold cyan]Model Path:[/bold cyan]       {BEST_PT_PATH}\n'
    f'[bold cyan]Severity Dir:[/bold cyan]     {SEVERITY_DIR}',
    title='📂 Project Configuration'
))

---
## 5. Model Loading (Reused from Notebook 08)
Load the trained `best.pt` model **once**. No retraining is performed.

In [ ]:
def load_model(model_path: Path, device: str = 'cpu') -> YOLO:
    """Load a trained YOLO model from checkpoint.

    Args:
        model_path: Path to the .pt model weights file.
        device: Compute device string ('cuda:0', 'mps', 'cpu').

    Returns:
        Loaded YOLO model ready for inference.

    Raises:
        FileNotFoundError: If the model file does not exist.
        RuntimeError: If the model fails to load.
    """
    if not model_path.exists():
        fallback = model_path.parent / 'train_logs' / 'PlasticSense_YOLOv11' / 'weights' / 'best.pt'
        if fallback.exists():
            shutil.copy2(fallback, model_path)
            console.print(f'[yellow]⚠ Copied best.pt from train_logs to {model_path}[/yellow]')
        else:
            raise FileNotFoundError(
                f'Model not found at {model_path}. Run Notebook 06 (Training) first.'
            )

    console.print(f'[cyan]Loading model from {model_path}...[/cyan]')

    try:
        model = YOLO(str(model_path))
    except Exception as e:
        raise RuntimeError(f'Failed to load model: {e}')

    model_size_mb = model_path.stat().st_size / (1024 * 1024)
    class_names = model.names
    num_classes = len(class_names)

    table = Table(title='🧠 Model Verification', show_header=True)
    table.add_column('Property', style='cyan', min_width=22)
    table.add_column('Value', justify='right', style='bold')
    table.add_row('Model Path', str(model_path.name))
    table.add_row('Model Size', f'{model_size_mb:.2f} MB')
    table.add_row('Number of Classes', str(num_classes))
    table.add_row('Class Names', ', '.join(class_names.values()))
    table.add_row('Status', '[bold green]✔ Loaded Successfully[/bold green]')
    console.print(table)

    logger.info(f'Model loaded: {model_path.name}, {num_classes} classes, {model_size_mb:.2f} MB')
    return model


# ── Load model ONCE ──
model = load_model(BEST_PT_PATH, DEVICE)
CLASS_NAMES: Dict[int, str] = model.names
NUM_CLASSES: int = len(CLASS_NAMES)

---
## 6. YOLO Inference Function (Reused from Notebook 08)
Run YOLO inference on a single image and extract detections.

In [ ]:
def predict_image(
    model: YOLO,
    image_path: Union[str, Path],
    conf_threshold: float = 0.25,
    iou_threshold: float = 0.5
) -> Dict[str, Any]:
    """Run YOLO inference on a single image and return structured results.

    Reused from Notebook 08 — predict_image().

    Args:
        model: Loaded YOLO model.
        image_path: Path to the input image.
        conf_threshold: Minimum confidence score for detections.
        iou_threshold: IoU threshold for NMS.

    Returns:
        Dictionary containing detections, summary, and timing info.
    """
    image_path = Path(image_path)

    # ── Validate ──
    if not image_path.exists():
        logger.warning(f'Image not found: {image_path}')
        return _error_result(image_path.name, 'Image not found')

    if image_path.suffix.lower() not in SUPPORTED_EXTENSIONS:
        logger.warning(f'Unsupported format: {image_path.name}')
        return _error_result(image_path.name, 'Unsupported image format')

    try:
        img = cv2.imread(str(image_path))
        if img is None:
            return _error_result(image_path.name, 'Corrupted image')
    except Exception as e:
        return _error_result(image_path.name, f'Read error: {e}')

    h, w = img.shape[:2]
    image_size = f'{w}x{h}'

    # ── Run inference ──
    start_time = time.perf_counter()
    results = model.predict(
        source=str(image_path),
        conf=conf_threshold,
        iou=iou_threshold,
        verbose=False
    )
    inference_time = (time.perf_counter() - start_time) * 1000  # ms

    # ── Extract detections ──
    detections: List[Dict[str, Any]] = []
    class_counter: Counter = Counter()
    confidences: List[float] = []

    if len(results) > 0 and results[0].boxes is not None:
        boxes = results[0].boxes
        for i, box in enumerate(boxes):
            cls_id = int(box.cls.item())
            conf = float(box.conf.item())
            xyxy = box.xyxy[0].cpu().numpy().tolist()
            x1, y1, x2, y2 = xyxy
            bbox_xywh = [
                round(x1, 2), round(y1, 2),
                round(x2 - x1, 2), round(y2 - y1, 2)
            ]

            class_name = model.names.get(cls_id, f'class_{cls_id}')
            class_counter[class_name] += 1
            confidences.append(conf)

            detections.append({
                'id': i + 1,
                'class': class_name,
                'class_id': cls_id,
                'confidence': round(conf, 4),
                'bbox': bbox_xywh
            })

    total_objects = len(detections)
    plastic_types = len(class_counter)
    avg_confidence = round(float(np.mean(confidences)), 4) if confidences else 0.0
    dominant_type = class_counter.most_common(1)[0][0] if class_counter else 'N/A'

    return {
        'image_name': image_path.name,
        'image_size': image_size,
        'image_width': w,
        'image_height': h,
        'status': 'success',
        'detections': detections,
        'summary': {
            'total_objects': total_objects,
            'plastic_types': plastic_types,
            'average_confidence': avg_confidence,
            'dominant_type': dominant_type,
            'objects_per_class': dict(class_counter),
            'inference_time_ms': round(inference_time, 2)
        }
    }


def _error_result(image_name: str, error_msg: str) -> Dict[str, Any]:
    """Return a standardized error result dict."""
    return {
        'image_name': image_name,
        'image_size': 'N/A',
        'image_width': 0,
        'image_height': 0,
        'status': 'error',
        'error': error_msg,
        'detections': [],
        'summary': {
            'total_objects': 0,
            'plastic_types': 0,
            'average_confidence': 0.0,
            'dominant_type': 'N/A',
            'objects_per_class': {},
            'inference_time_ms': 0.0
        }
    }


console.print('[bold green]✔ Inference function ready: predict_image()[/bold green]')

---
## 7. Hazard Weight Configuration (Step 3)
Assign environmental hazard weights to each plastic class. Higher weight = greater environmental threat.

In [ ]:
# ══════════════════════════════════════════════════════════════
# Hazard Weight Configuration
# ══════════════════════════════════════════════════════════════
# Weights reflect environmental persistence, toxicity, and
# decomposition difficulty. Scale: 1 (least) to 4 (most hazardous).
#
# Rationale:
#   - Plastic Bottle / Cap: Recyclable → weight 1
#   - Bags, Wrappers, Food Containers: Common litter → weight 2
#   - Styrofoam: Fragments into microplastics → weight 3
#   - Multilayer Packaging: Non-recyclable → weight 4
# ══════════════════════════════════════════════════════════════

HAZARD_WEIGHTS: Dict[str, int] = {
    'plastic_bottle':         1,
    'plastic_cap':            1,
    'plastic_bag':            2,
    'wrapper':                2,
    'food_container':         2,
    'styrofoam':              3,
    'multilayer_packaging':   4,
    'other_plastic':          2,
}

# Maximum possible hazard weight (for normalization)
MAX_HAZARD_WEIGHT = max(HAZARD_WEIGHTS.values())

# Display the hazard weight table
hw_table = Table(
    title='☣️ Hazard Weight Configuration',
    show_header=True, header_style='bold red'
)
hw_table.add_column('Plastic Class', style='cyan', min_width=24)
hw_table.add_column('Hazard Weight', justify='center', style='bold')
hw_table.add_column('Risk Level', justify='center')

risk_labels = {1: '[green]Low[/green]', 2: '[yellow]Moderate[/yellow]',
               3: '[dark_orange]High[/dark_orange]', 4: '[bold red]Critical[/bold red]'}

for cls_name, weight in HAZARD_WEIGHTS.items():
    hw_table.add_row(cls_name, str(weight), risk_labels.get(weight, str(weight)))

console.print(hw_table)
logger.info(f'Hazard weights configured for {len(HAZARD_WEIGHTS)} classes.')

---
## 8. Severity Classification Thresholds
Define the mapping from severity score to severity level and badge.

In [ ]:
# ══════════════════════════════════════════════════════════════
# Severity Classification Thresholds
# ══════════════════════════════════════════════════════════════

SEVERITY_LEVELS: List[Dict[str, Any]] = [
    {'min': 0,  'max': 25,  'level': 'Low',      'badge': '🟢', 'color': '#2ecc71'},
    {'min': 26, 'max': 50,  'level': 'Medium',   'badge': '🟡', 'color': '#f1c40f'},
    {'min': 51, 'max': 75,  'level': 'High',     'badge': '🟠', 'color': '#e67e22'},
    {'min': 76, 'max': 100, 'level': 'Critical', 'badge': '🔴', 'color': '#e74c3c'},
]

# Severity formula weights
W_COUNT: float = 0.35   # Weight for normalized object count
W_DENSITY: float = 0.30  # Weight for plastic density score
W_HAZARD: float = 0.25   # Weight for hazard score
W_WATER: float = 0.10    # Weight for waterbody proximity score

# Normalization ceilings (configurable)
MAX_EXPECTED_OBJECTS: int = 50   # Objects above this yield score = 100
MAX_EXPECTED_DENSITY: float = 60.0  # Density % above this yields score = 100


def classify_severity(score: float) -> Dict[str, Any]:
    """Classify a severity score into a level with badge and color.

    Args:
        score: Severity score (0-100).

    Returns:
        Dict with 'level', 'badge', and 'color' keys.
    """
    score = max(0.0, min(100.0, score))
    for level_info in SEVERITY_LEVELS:
        if level_info['min'] <= score <= level_info['max']:
            return {
                'level': level_info['level'],
                'badge': level_info['badge'],
                'color': level_info['color']
            }
    return {'level': 'Critical', 'badge': '🔴', 'color': '#e74c3c'}


# Display thresholds
sev_table = Table(
    title='🚦 Severity Classification Thresholds',
    show_header=True, header_style='bold magenta'
)
sev_table.add_column('Score Range', style='cyan', justify='center')
sev_table.add_column('Level', justify='center', style='bold')
sev_table.add_column('Badge', justify='center')

for lvl in SEVERITY_LEVELS:
    sev_table.add_row(
        f"{lvl['min']}–{lvl['max']}",
        lvl['level'],
        lvl['badge']
    )

console.print(sev_table)

console.print(Panel.fit(
    f'[bold cyan]Severity Formula:[/bold cyan]\n'
    f'  score = {W_COUNT} × Object Count (norm)\n'
    f'        + {W_DENSITY} × Density Score (norm)\n'
    f'        + {W_HAZARD} × Hazard Score (norm)\n'
    f'        + {W_WATER} × Waterbody Score',
    title='🧮 Formula Weights'
))
logger.info('Severity thresholds and formula weights configured.')

---
## 9. Core Severity Functions (Steps 2–7)
These reusable functions will be directly copied into the FastAPI backend.

In [ ]:
def calculate_density(
    detections: List[Dict[str, Any]],
    image_width: int,
    image_height: int
) -> Dict[str, Any]:
    """Calculate plastic density as the percentage of image covered by detections.

    Args:
        detections: List of detection dicts with 'bbox' as [x, y, w, h].
        image_width: Width of the source image in pixels.
        image_height: Height of the source image in pixels.

    Returns:
        Dict with total_bbox_area, image_area, coverage_percentage, and density_score.
    """
    image_area = image_width * image_height

    if image_area == 0 or not detections:
        return {
            'total_bbox_area': 0,
            'image_area': image_area,
            'coverage_percentage': 0.0,
            'density_score': 0.0
        }

    total_bbox_area = 0
    for det in detections:
        _, _, bw, bh = det['bbox']
        total_bbox_area += bw * bh

    coverage_pct = (total_bbox_area / image_area) * 100.0
    # Normalize to 0-100 scale
    density_score = min(coverage_pct / MAX_EXPECTED_DENSITY * 100.0, 100.0)

    return {
        'total_bbox_area': round(total_bbox_area, 2),
        'image_area': image_area,
        'coverage_percentage': round(coverage_pct, 2),
        'density_score': round(density_score, 2)
    }


console.print('[bold green]✔ Density function defined: calculate_density()[/bold green]')

In [ ]:
def calculate_hazard(
    detections: List[Dict[str, Any]],
    hazard_weights: Dict[str, int] = HAZARD_WEIGHTS
) -> Dict[str, Any]:
    """Calculate the weighted hazard score from detected plastic types.

    Hazard Score = sum(count_i × weight_i) for each class i,
    then normalized to 0-100 scale.

    Args:
        detections: List of detection dicts with 'class' key.
        hazard_weights: Mapping from class name to hazard weight.

    Returns:
        Dict with raw_score, max_possible, normalized hazard_score, and per-class breakdown.
    """
    if not detections:
        return {
            'raw_score': 0,
            'max_possible_score': 0,
            'hazard_score': 0.0,
            'per_class_hazard': {}
        }

    per_class_hazard: Dict[str, Dict[str, Any]] = {}
    raw_score = 0

    class_counter: Counter = Counter()
    for det in detections:
        class_counter[det['class']] += 1

    for cls_name, count in class_counter.items():
        weight = hazard_weights.get(cls_name, 2)  # Default weight = 2
        cls_hazard = count * weight
        raw_score += cls_hazard
        per_class_hazard[cls_name] = {
            'count': count,
            'weight': weight,
            'hazard_contribution': cls_hazard
        }

    # Max possible = all objects with max weight
    max_possible = len(detections) * MAX_HAZARD_WEIGHT
    # Normalize to 0-100
    hazard_score = (raw_score / max_possible * 100.0) if max_possible > 0 else 0.0

    return {
        'raw_score': raw_score,
        'max_possible_score': max_possible,
        'hazard_score': round(hazard_score, 2),
        'per_class_hazard': per_class_hazard
    }


console.print('[bold green]✔ Hazard function defined: calculate_hazard()[/bold green]')

In [ ]:
def check_waterbody_proximity(
    latitude: Optional[float] = None,
    longitude: Optional[float] = None,
    proximity_threshold_km: float = 1.0
) -> Dict[str, Any]:
    """Check proximity to the nearest waterbody.

    This function uses mock/sample GIS data for demonstration.
    The implementation is modular and can be replaced with
    OpenStreetMap Overpass API or PostGIS queries in production.

    Args:
        latitude: GPS latitude of the image location.
        longitude: GPS longitude of the image location.
        proximity_threshold_km: Distance threshold to classify as 'near water'.

    Returns:
        Dict with distance_km, near_water, waterbody_name, and waterbody_score.
    """
    # ── Mock waterbody database ──
    # In production, replace with OpenStreetMap Overpass API or PostGIS
    WATERBODY_DATABASE: List[Dict[str, Any]] = [
        {'name': 'Yamuna River',        'lat': 28.6139, 'lon': 77.2090, 'type': 'river'},
        {'name': 'Ganges River',         'lat': 25.3176, 'lon': 82.9739, 'type': 'river'},
        {'name': 'Hussain Sagar Lake',   'lat': 17.4239, 'lon': 78.4738, 'type': 'lake'},
        {'name': 'Dal Lake',             'lat': 34.1100, 'lon': 74.8600, 'type': 'lake'},
        {'name': 'Marina Beach Coast',   'lat': 13.0500, 'lon': 80.2824, 'type': 'coast'},
        {'name': 'Juhu Beach Coast',     'lat': 19.0988, 'lon': 72.8267, 'type': 'coast'},
        {'name': 'Powai Lake',           'lat': 19.1275, 'lon': 72.9060, 'type': 'lake'},
        {'name': 'Chilika Lake',         'lat': 19.6917, 'lon': 85.3183, 'type': 'lake'},
        {'name': 'Godavari River',       'lat': 16.5417, 'lon': 81.5222, 'type': 'river'},
        {'name': 'Cauvery River',        'lat': 11.9416, 'lon': 79.8083, 'type': 'river'},
    ]

    # ── No GPS provided ──
    if latitude is None or longitude is None:
        logger.info('No GPS coordinates provided. Waterbody check skipped.')
        return {
            'distance_km': None,
            'near_water': False,
            'waterbody_name': 'N/A',
            'waterbody_type': 'N/A',
            'waterbody_score': 0.0
        }

    # ── Haversine distance ──
    def haversine(lat1: float, lon1: float, lat2: float, lon2: float) -> float:
        """Calculate distance in km between two GPS coordinates."""
        R = 6371.0  # Earth radius in km
        lat1_r, lat2_r = math.radians(lat1), math.radians(lat2)
        dlat = math.radians(lat2 - lat1)
        dlon = math.radians(lon2 - lon1)
        a = (math.sin(dlat / 2) ** 2 +
             math.cos(lat1_r) * math.cos(lat2_r) * math.sin(dlon / 2) ** 2)
        c = 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))
        return R * c

    # ── Find nearest waterbody ──
    nearest_distance = float('inf')
    nearest_waterbody = WATERBODY_DATABASE[0]

    for wb in WATERBODY_DATABASE:
        dist = haversine(latitude, longitude, wb['lat'], wb['lon'])
        if dist < nearest_distance:
            nearest_distance = dist
            nearest_waterbody = wb

    near_water = nearest_distance <= proximity_threshold_km

    # ── Calculate waterbody score (0-100) ──
    # Score inversely proportional to distance
    # 0 km = 100, >= 10 km = 0, linear decay
    max_influence_km = 10.0
    if nearest_distance >= max_influence_km:
        waterbody_score = 0.0
    else:
        waterbody_score = (1.0 - nearest_distance / max_influence_km) * 100.0

    logger.info(
        f'Nearest waterbody: {nearest_waterbody["name"]} '
        f'({nearest_distance:.2f} km), near_water={near_water}'
    )

    return {
        'distance_km': round(nearest_distance, 2),
        'near_water': near_water,
        'waterbody_name': nearest_waterbody['name'],
        'waterbody_type': nearest_waterbody['type'],
        'waterbody_score': round(waterbody_score, 2)
    }


console.print('[bold green]✔ Waterbody proximity function defined: check_waterbody_proximity()[/bold green]')

In [ ]:
def calculate_severity(
    detection_result: Dict[str, Any],
    latitude: Optional[float] = None,
    longitude: Optional[float] = None,
    timestamp: Optional[str] = None
) -> Dict[str, Any]:
    """Calculate the overall environmental severity score.

    Combines object count, density, hazard weights, and waterbody
    proximity into a single 0-100 severity score.

    Args:
        detection_result: Output from predict_image().
        latitude: GPS latitude of the image location.
        longitude: GPS longitude of the image location.
        timestamp: Optional ISO timestamp string.

    Returns:
        Comprehensive severity assessment dict.
    """
    if detection_result.get('status') != 'success':
        return {
            'image_name': detection_result.get('image_name', 'unknown'),
            'status': 'error',
            'error': detection_result.get('error', 'Detection failed'),
            'severity': {'score': 0, 'level': 'N/A', 'badge': '⚪'},
            'plastic_summary': {},
            'hazard': {},
            'location': {}
        }

    detections = detection_result['detections']
    summary = detection_result['summary']
    img_w = detection_result.get('image_width', 640)
    img_h = detection_result.get('image_height', 640)

    # ── Step 2: Calculate density ──
    density_info = calculate_density(detections, img_w, img_h)

    # ── Step 4: Calculate hazard score ──
    hazard_info = calculate_hazard(detections)

    # ── Step 5: Check waterbody proximity ──
    water_info = check_waterbody_proximity(latitude, longitude)

    # ── Step 6: Calculate severity score ──
    total_objects = summary['total_objects']

    # Normalize object count (0-100)
    count_score = min(total_objects / MAX_EXPECTED_OBJECTS * 100.0, 100.0)

    # Gather component scores
    density_score = density_info['density_score']
    hazard_score = hazard_info['hazard_score']
    waterbody_score = water_info['waterbody_score']

    # Weighted severity formula
    severity_score = (
        W_COUNT * count_score
        + W_DENSITY * density_score
        + W_HAZARD * hazard_score
        + W_WATER * waterbody_score
    )
    severity_score = round(max(0.0, min(100.0, severity_score)), 2)

    # ── Step 7: Classify severity ──
    severity_class = classify_severity(severity_score)

    # ── Step 8: Build output JSON ──
    result = {
        'image_name': detection_result['image_name'],
        'image_size': detection_result['image_size'],
        'timestamp': timestamp or datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
        'status': 'success',

        'severity': {
            'score': severity_score,
            'level': severity_class['level'],
            'badge': severity_class['badge'],
            'color': severity_class['color']
        },

        'plastic_summary': {
            'total_objects': total_objects,
            'plastic_types': summary['plastic_types'],
            'dominant_class': summary['dominant_type'],
            'average_confidence': summary['average_confidence'],
            'objects_per_class': summary.get('objects_per_class', {}),
            'density': density_info['coverage_percentage']
        },

        'density': {
            'total_bbox_area': density_info['total_bbox_area'],
            'image_area': density_info['image_area'],
            'coverage_percentage': density_info['coverage_percentage'],
            'density_score': density_info['density_score']
        },

        'hazard': {
            'score': hazard_info['hazard_score'],
            'raw_score': hazard_info['raw_score'],
            'max_possible': hazard_info['max_possible_score'],
            'per_class_hazard': hazard_info['per_class_hazard']
        },

        'location': {
            'latitude': latitude,
            'longitude': longitude,
            'near_water': water_info['near_water'],
            'waterbody_name': water_info['waterbody_name'],
            'waterbody_type': water_info['waterbody_type'],
            'distance_km': water_info['distance_km'],
            'waterbody_score': water_info['waterbody_score']
        },

        'component_scores': {
            'count_score': round(count_score, 2),
            'density_score': density_score,
            'hazard_score': hazard_score,
            'waterbody_score': waterbody_score
        },

        'formula_weights': {
            'w_count': W_COUNT,
            'w_density': W_DENSITY,
            'w_hazard': W_HAZARD,
            'w_water': W_WATER
        }
    }

    logger.info(
        f'Severity assessed for {detection_result["image_name"]}: '
        f'score={severity_score}, level={severity_class["level"]}'
    )

    return result


console.print('[bold green]✔ Severity function defined: calculate_severity()[/bold green]')

---
## 10. Report Generation Functions (Steps 8–10)
Generate structured JSON and CSV reports for each severity assessment.

In [ ]:
def save_json(
    severity_result: Dict[str, Any],
    output_dir: Path
) -> Path:
    """Save severity assessment result as JSON.

    Args:
        severity_result: Output from calculate_severity().
        output_dir: Directory to save the JSON file.

    Returns:
        Path to the saved JSON file.
    """
    output_dir.mkdir(parents=True, exist_ok=True)
    image_stem = Path(severity_result['image_name']).stem
    json_path = output_dir / f'{image_stem}_severity.json'

    with open(json_path, 'w') as f:
        json.dump(severity_result, f, indent=4, default=str)

    logger.info(f'Severity JSON saved: {json_path.name}')
    return json_path


def save_csv(
    severity_results: List[Dict[str, Any]],
    output_dir: Path,
    filename: str = 'severity_report.csv'
) -> Path:
    """Save all severity results as a flat CSV file.

    Args:
        severity_results: List of severity result dicts.
        output_dir: Directory to save the CSV file.
        filename: Name of the CSV file.

    Returns:
        Path to the saved CSV file.
    """
    output_dir.mkdir(parents=True, exist_ok=True)
    csv_path = output_dir / filename

    rows: List[Dict[str, Any]] = []
    for res in severity_results:
        sev = res.get('severity', {})
        ps = res.get('plastic_summary', {})
        haz = res.get('hazard', {})
        loc = res.get('location', {})
        comp = res.get('component_scores', {})

        rows.append({
            'image_name': res.get('image_name', ''),
            'timestamp': res.get('timestamp', ''),
            'status': res.get('status', 'error'),
            'severity_score': sev.get('score', 0),
            'severity_level': sev.get('level', 'N/A'),
            'severity_badge': sev.get('badge', ''),
            'total_objects': ps.get('total_objects', 0),
            'plastic_types': ps.get('plastic_types', 0),
            'dominant_class': ps.get('dominant_class', 'N/A'),
            'average_confidence': ps.get('average_confidence', 0.0),
            'coverage_pct': ps.get('density', 0.0),
            'hazard_score': haz.get('score', 0.0),
            'hazard_raw': haz.get('raw_score', 0),
            'latitude': loc.get('latitude', ''),
            'longitude': loc.get('longitude', ''),
            'near_water': loc.get('near_water', False),
            'waterbody_name': loc.get('waterbody_name', 'N/A'),
            'distance_km': loc.get('distance_km', ''),
            'count_score': comp.get('count_score', 0),
            'density_score': comp.get('density_score', 0),
            'waterbody_score': comp.get('waterbody_score', 0)
        })

    df = pd.DataFrame(rows)
    df.to_csv(csv_path, index=False)
    logger.info(f'Severity CSV saved: {csv_path.name} ({len(rows)} rows)')
    return csv_path


def generate_report(
    severity_results: List[Dict[str, Any]],
    output_dir: Path
) -> Path:
    """Generate a batch severity summary report as JSON.

    Args:
        severity_results: List of severity result dicts.
        output_dir: Directory to save the report.

    Returns:
        Path to the saved report JSON.
    """
    output_dir.mkdir(parents=True, exist_ok=True)

    successful = [r for r in severity_results if r.get('status') == 'success']

    if not successful:
        report = {
            'project': 'PlasticSense AI',
            'notebook': '09_Severity_Engine',
            'timestamp': datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
            'total_images': len(severity_results),
            'successful': 0,
            'message': 'No successful assessments'
        }
    else:
        scores = [r['severity']['score'] for r in successful]
        levels = Counter(r['severity']['level'] for r in successful)
        total_objects = sum(r['plastic_summary']['total_objects'] for r in successful)

        report = {
            'project': 'PlasticSense AI',
            'notebook': '09_Severity_Engine',
            'timestamp': datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
            'total_images': len(severity_results),
            'successful': len(successful),
            'total_plastic_objects': total_objects,
            'average_severity_score': round(float(np.mean(scores)), 2),
            'min_severity_score': round(float(np.min(scores)), 2),
            'max_severity_score': round(float(np.max(scores)), 2),
            'severity_distribution': dict(levels),
            'most_common_severity': levels.most_common(1)[0][0] if levels else 'N/A'
        }

    report_path = output_dir / 'severity_batch_report.json'
    with open(report_path, 'w') as f:
        json.dump(report, f, indent=4, default=str)

    logger.info(f'Batch severity report saved: {report_path.name}')
    return report_path


console.print('[bold green]✔ Report functions defined: save_json(), save_csv(), generate_report()[/bold green]')

---
## 11. Visualization — Severity Dashboard (Step 11)
Generate a visual severity dashboard showing detection, density, hazard, and severity badge.

In [ ]:
# Color palette for bounding boxes (BGR → RGB for matplotlib)
CLASS_COLORS_RGB: Dict[int, Tuple[float, float, float]] = {
    0: (1.0,  0.647, 0.0),    # plastic_bottle  → Orange
    1: (1.0,  0.078, 0.576),   # plastic_bag     → Pink
    2: (0.498, 1.0,  0.0),     # wrapper         → Spring Green
    3: (0.0,  0.749, 1.0),     # styrofoam       → Deep Sky Blue
    4: (1.0,  1.0,  0.0),      # plastic_cap     → Yellow
    5: (0.0,  0.0,  1.0),      # food_container  → Blue
    6: (1.0,  0.412, 0.706),   # multilayer_pkg  → Hot Pink
    7: (1.0,  0.843, 0.0),     # other_plastic   → Gold
}


def visualize_severity(
    image_path: Union[str, Path],
    severity_result: Dict[str, Any],
    detection_result: Dict[str, Any],
    save_dir: Optional[Path] = None,
    show: bool = True
) -> None:
    """Generate a multi-panel severity dashboard for a single image.

    Panels:
        1. Original Image with Detections
        2. Plastic Count Bar Chart
        3. Component Scores Radar-style Bar
        4. Severity Badge Display

    Args:
        image_path: Path to the original image.
        severity_result: Output from calculate_severity().
        detection_result: Output from predict_image().
        save_dir: Optional directory to save the plot.
        show: Whether to display inline.
    """
    image_path = Path(image_path)
    img = cv2.imread(str(image_path))
    if img is None:
        logger.error(f'Cannot read image for visualization: {image_path}')
        return

    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]

    detections = detection_result.get('detections', [])
    sev = severity_result.get('severity', {})
    ps = severity_result.get('plastic_summary', {})
    comp = severity_result.get('component_scores', {})
    haz = severity_result.get('hazard', {})

    fig, axes = plt.subplots(2, 2, figsize=(18, 14))
    fig.suptitle(
        f'PlasticSense AI — Severity Assessment: {image_path.name}',
        fontsize=16, fontweight='bold', y=0.98
    )

    # ── Panel 1: Image with Detections ──
    ax1 = axes[0, 0]
    ax1.imshow(img_rgb)

    for det in detections:
        x, y, bw, bh = det['bbox']
        cls_id = det.get('class_id', 0)
        color = CLASS_COLORS_RGB.get(cls_id, (0.5, 0.5, 0.5))
        rect = plt.Rectangle((x, y), bw, bh,
                             linewidth=2, edgecolor=color, facecolor='none')
        ax1.add_patch(rect)
        label = f"{det['class']} {det['confidence']:.2f}"
        ax1.text(x, max(y - 4, 10), label, fontsize=7, color='white',
                 bbox=dict(boxstyle='round,pad=0.2', facecolor=color, alpha=0.8))

    ax1.set_title(f'Detection ({len(detections)} objects)', fontweight='bold')
    ax1.axis('off')

    # ── Panel 2: Objects Per Class ──
    ax2 = axes[0, 1]
    opc = ps.get('objects_per_class', {})
    if opc:
        classes = list(opc.keys())
        counts = list(opc.values())
        colors = [CLASS_COLORS_RGB.get(list(CLASS_NAMES.values()).index(c), (0.5, 0.5, 0.5))
                  if c in CLASS_NAMES.values() else (0.5, 0.5, 0.5)
                  for c in classes]
        bars = ax2.barh(classes, counts, color=colors, edgecolor='black', linewidth=0.5)
        for bar, count in zip(bars, counts):
            ax2.text(bar.get_width() + 0.2, bar.get_y() + bar.get_height()/2,
                     str(count), va='center', fontweight='bold', fontsize=10)
        ax2.set_xlabel('Count')
        ax2.set_title('Plastic Count by Class', fontweight='bold')
    else:
        ax2.text(0.5, 0.5, 'No Detections', ha='center', va='center',
                 fontsize=16, color='gray', transform=ax2.transAxes)
        ax2.set_title('Plastic Count by Class', fontweight='bold')

    # ── Panel 3: Component Scores ──
    ax3 = axes[1, 0]
    score_names = ['Object Count', 'Density', 'Hazard', 'Waterbody']
    score_values = [
        comp.get('count_score', 0),
        comp.get('density_score', 0),
        comp.get('hazard_score', 0),
        comp.get('waterbody_score', 0)
    ]
    bar_colors = ['#3498db', '#2ecc71', '#e74c3c', '#9b59b6']
    bars = ax3.bar(score_names, score_values, color=bar_colors,
                   edgecolor='black', linewidth=0.5)
    for bar, val in zip(bars, score_values):
        ax3.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                 f'{val:.1f}', ha='center', fontweight='bold', fontsize=10)
    ax3.set_ylim(0, 110)
    ax3.set_ylabel('Score (0-100)')
    ax3.set_title('Component Scores', fontweight='bold')
    ax3.axhline(y=50, color='orange', linestyle='--', alpha=0.4, label='Medium threshold')

    # ── Panel 4: Severity Badge ──
    ax4 = axes[1, 1]
    ax4.axis('off')

    sev_score = sev.get('score', 0)
    sev_level = sev.get('level', 'N/A')
    sev_badge = sev.get('badge', '⚪')
    sev_color = sev.get('color', '#95a5a6')

    # Large badge circle
    circle = plt.Circle((0.5, 0.6), 0.28, color=sev_color, alpha=0.15,
                        transform=ax4.transAxes)
    ax4.add_patch(circle)

    ax4.text(0.5, 0.72, sev_badge, ha='center', va='center',
             fontsize=48, transform=ax4.transAxes)
    ax4.text(0.5, 0.52, f'{sev_score:.1f}', ha='center', va='center',
             fontsize=36, fontweight='bold', color=sev_color,
             transform=ax4.transAxes)
    ax4.text(0.5, 0.38, sev_level.upper(), ha='center', va='center',
             fontsize=22, fontweight='bold', color=sev_color,
             transform=ax4.transAxes)

    # Stats below badge
    stats_text = (
        f"Objects: {ps.get('total_objects', 0)}  |  "
        f"Density: {ps.get('density', 0):.1f}%  |  "
        f"Hazard: {haz.get('score', 0):.1f}"
    )
    ax4.text(0.5, 0.18, stats_text, ha='center', va='center',
             fontsize=11, color='#555', transform=ax4.transAxes)

    loc_info = severity_result.get('location', {})
    if loc_info.get('latitude') is not None:
        loc_text = (
            f"📍 ({loc_info['latitude']}, {loc_info['longitude']})  |  "
            f"💧 {loc_info.get('waterbody_name', 'N/A')} "
            f"({loc_info.get('distance_km', '?')} km)"
        )
        ax4.text(0.5, 0.08, loc_text, ha='center', va='center',
                 fontsize=9, color='#777', transform=ax4.transAxes)

    ax4.set_title('Severity Assessment', fontweight='bold')

    plt.tight_layout(rect=[0, 0, 1, 0.96])

    # ── Save ──
    if save_dir is not None:
        save_dir.mkdir(parents=True, exist_ok=True)
        stem = Path(severity_result['image_name']).stem
        save_path = save_dir / f'{stem}_severity_dashboard.png'
        fig.savefig(save_path, dpi=150, bbox_inches='tight')
        logger.info(f'Severity dashboard saved: {save_path.name}')

    if show:
        plt.show()
    plt.close(fig)


console.print('[bold green]✔ Visualization function defined: visualize_severity()[/bold green]')

---
## 12. End-to-End Pipeline — Assess Single Image
Combine detection → density → hazard → waterbody → severity into one call.

In [ ]:
def assess_severity(
    model: YOLO,
    image_path: Union[str, Path],
    latitude: Optional[float] = None,
    longitude: Optional[float] = None,
    timestamp: Optional[str] = None,
    save_dir: Optional[Path] = None,
    visualize: bool = True
) -> Dict[str, Any]:
    """Full severity assessment pipeline for a single image.

    Steps:
        1. Run YOLO inference
        2. Calculate density, hazard, waterbody proximity
        3. Compute severity score and classify
        4. Generate visualization and save reports

    Args:
        model: Loaded YOLO model.
        image_path: Path to the input image.
        latitude: GPS latitude.
        longitude: GPS longitude.
        timestamp: Optional ISO timestamp.
        save_dir: Optional base directory for outputs.
        visualize: Whether to display the severity dashboard.

    Returns:
        Complete severity result dict.
    """
    image_path = Path(image_path)
    console.print(f'\n[bold cyan]─── Severity Assessment: {image_path.name} ───[/bold cyan]')

    # Step 1: YOLO Inference
    console.print('[cyan]Step 1: Running YOLO inference...[/cyan]')
    detection_result = predict_image(model, image_path)

    if detection_result.get('status') != 'success':
        console.print(f'[bold red]✖ Detection failed: {detection_result.get("error")}[/bold red]')
        return detection_result

    summary = detection_result['summary']
    console.print(
        f'  ✔ Detected {summary["total_objects"]} objects, '
        f'{summary["plastic_types"]} types, '
        f'{summary["average_confidence"]:.4f} avg conf'
    )

    # Steps 2-7: Calculate severity
    console.print('[cyan]Step 2-7: Computing severity score...[/cyan]')
    severity_result = calculate_severity(
        detection_result, latitude, longitude, timestamp
    )

    sev = severity_result['severity']

    # Display severity table
    result_table = Table(
        title=f'{sev["badge"]} Severity Result',
        show_header=True, header_style='bold'
    )
    result_table.add_column('Metric', style='cyan', min_width=24)
    result_table.add_column('Value', justify='right', style='bold')

    ps = severity_result['plastic_summary']
    comp = severity_result['component_scores']
    haz = severity_result['hazard']
    loc = severity_result['location']

    result_table.add_row('Severity Score', f'{sev["score"]:.2f}')
    result_table.add_row('Severity Level', f'{sev["badge"]} {sev["level"]}')
    result_table.add_row('─' * 20, '─' * 15)
    result_table.add_row('Total Objects', str(ps['total_objects']))
    result_table.add_row('Plastic Types', str(ps['plastic_types']))
    result_table.add_row('Dominant Class', ps['dominant_class'])
    result_table.add_row('Coverage %', f'{ps["density"]:.2f}%')
    result_table.add_row('─' * 20, '─' * 15)
    result_table.add_row('Count Score', f'{comp["count_score"]:.2f}')
    result_table.add_row('Density Score', f'{comp["density_score"]:.2f}')
    result_table.add_row('Hazard Score', f'{comp["hazard_score"]:.2f}')
    result_table.add_row('Waterbody Score', f'{comp["waterbody_score"]:.2f}')

    if loc.get('latitude') is not None:
        result_table.add_row('─' * 20, '─' * 15)
        result_table.add_row('GPS', f'({loc["latitude"]}, {loc["longitude"]})')
        result_table.add_row('Near Water', str(loc['near_water']))
        result_table.add_row('Nearest Waterbody', f'{loc["waterbody_name"]} ({loc["distance_km"]} km)')

    console.print(result_table)

    # Save outputs
    if save_dir is not None:
        save_json(severity_result, save_dir / 'json')

    # Visualize
    if visualize:
        visualize_severity(
            image_path, severity_result, detection_result,
            save_dir=save_dir / 'plots' if save_dir else None,
            show=True
        )

    return severity_result


console.print('[bold green]✔ Pipeline function defined: assess_severity()[/bold green]')

---
## 13. Run Severity Assessment on Test Images
Execute the full severity pipeline on sample test images with mock GPS data.

In [ ]:
# ── Sample GPS coordinates for demo (mock data) ──
# In production, GPS comes from image EXIF or user input.
SAMPLE_LOCATIONS: List[Dict[str, Any]] = [
    {'lat': 19.1200, 'lon': 72.9000, 'desc': 'Near Powai Lake, Mumbai'},
    {'lat': 28.6300, 'lon': 77.2200, 'desc': 'Near Yamuna River, Delhi'},
    {'lat': 13.0600, 'lon': 80.2800, 'desc': 'Near Marina Beach, Chennai'},
    {'lat': 22.5726, 'lon': 88.3639, 'desc': 'Kolkata, West Bengal'},
    {'lat': 17.4300, 'lon': 78.4700, 'desc': 'Near Hussain Sagar, Hyderabad'},
]

# ── Discover test images ──
all_severity_results: List[Dict[str, Any]] = []

if TEST_IMAGES.exists():
    test_files = sorted([
        f for f in TEST_IMAGES.iterdir()
        if f.suffix.lower() in SUPPORTED_EXTENSIONS
    ])

    # Process a manageable subset for demo
    MAX_DEMO_IMAGES = 10
    demo_files = test_files[:MAX_DEMO_IMAGES]

    console.print(f'[cyan]Processing {len(demo_files)} test images for severity assessment...[/cyan]')

    for idx, img_path in enumerate(tqdm(demo_files, desc='⚠️ Severity Assessment')):
        # Cycle through sample locations
        loc = SAMPLE_LOCATIONS[idx % len(SAMPLE_LOCATIONS)]

        result = assess_severity(
            model=model,
            image_path=img_path,
            latitude=loc['lat'],
            longitude=loc['lon'],
            save_dir=SEVERITY_DIR,
            visualize=(idx < 3)  # Show dashboard for first 3 only
        )
        all_severity_results.append(result)

    console.print(f'\n[bold green]✔ Severity assessment complete for {len(demo_files)} images.[/bold green]')
else:
    console.print('[bold red]✖ Test images directory not found. Skipping batch assessment.[/bold red]')
    console.print(f'[yellow]Expected: {TEST_IMAGES}[/yellow]')

---
## 14. Export Batch Results — CSV & JSON Reports (Steps 9–10)
Save all severity results and generate the batch summary report.

In [ ]:
if all_severity_results:
    # ── Save CSV ──
    csv_path = save_csv(all_severity_results, SEV_CSV_DIR, 'severity_report.csv')
    console.print(f'[green]✔ CSV report saved: {csv_path}[/green]')

    # ── Save batch JSON report ──
    report_path = generate_report(all_severity_results, SEV_REPORTS_DIR)
    console.print(f'[green]✔ Batch report saved: {report_path}[/green]')

    # ── Save combined severity JSON (all images) ──
    combined_json_path = SEV_REPORTS_DIR / 'severity_report.json'
    with open(combined_json_path, 'w') as f:
        json.dump(all_severity_results, f, indent=4, default=str)
    console.print(f'[green]✔ Combined JSON saved: {combined_json_path}[/green]')

    # ── Display batch summary table ──
    successful = [r for r in all_severity_results if r.get('status') == 'success']
    if successful:
        scores = [r['severity']['score'] for r in successful]
        levels = Counter(r['severity']['level'] for r in successful)

        batch_table = Table(
            title='📊 Batch Severity Summary',
            show_header=True, header_style='bold magenta'
        )
        batch_table.add_column('Metric', style='cyan', min_width=28)
        batch_table.add_column('Value', justify='right', style='bold')

        batch_table.add_row('Total Images', str(len(all_severity_results)))
        batch_table.add_row('Successful', str(len(successful)))
        batch_table.add_row('Average Severity Score', f'{np.mean(scores):.2f}')
        batch_table.add_row('Min Severity Score', f'{np.min(scores):.2f}')
        batch_table.add_row('Max Severity Score', f'{np.max(scores):.2f}')

        for level_name, count in sorted(levels.items()):
            batch_table.add_row(f'  {level_name}', str(count))

        console.print(batch_table)
else:
    console.print('[yellow]⚠ No severity results to export.[/yellow]')

---
## 15. Severity Distribution Visualization
Plot the distribution of severity levels across all processed images.

In [ ]:
def plot_severity_distribution(
    severity_results: List[Dict[str, Any]],
    save_dir: Optional[Path] = None
) -> None:
    """Plot severity score distribution and level breakdown."""
    successful = [r for r in severity_results if r.get('status') == 'success']
    if not successful:
        console.print('[yellow]⚠ No successful results to plot.[/yellow]')
        return

    scores = [r['severity']['score'] for r in successful]
    levels = [r['severity']['level'] for r in successful]
    level_counts = Counter(levels)

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    # ── Histogram ──
    ax1 = axes[0]
    ax1.hist(scores, bins=20, range=(0, 100), color='#3498db',
             edgecolor='black', alpha=0.8)
    # Severity zone backgrounds
    ax1.axvspan(0, 25, alpha=0.08, color='#2ecc71')
    ax1.axvspan(25, 50, alpha=0.08, color='#f1c40f')
    ax1.axvspan(50, 75, alpha=0.08, color='#e67e22')
    ax1.axvspan(75, 100, alpha=0.08, color='#e74c3c')
    ax1.axvline(np.mean(scores), color='red', linestyle='--', linewidth=2,
                label=f'Mean: {np.mean(scores):.1f}')
    ax1.set_xlabel('Severity Score')
    ax1.set_ylabel('Number of Images')
    ax1.set_title('Severity Score Distribution', fontweight='bold')
    ax1.legend()

    # ── Pie chart ──
    ax2 = axes[1]
    level_order = ['Low', 'Medium', 'High', 'Critical']
    level_colors = ['#2ecc71', '#f1c40f', '#e67e22', '#e74c3c']
    pie_labels = []
    pie_sizes = []
    pie_colors = []

    for lvl, clr in zip(level_order, level_colors):
        count = level_counts.get(lvl, 0)
        if count > 0:
            pie_labels.append(f'{lvl} ({count})')
            pie_sizes.append(count)
            pie_colors.append(clr)

    if pie_sizes:
        ax2.pie(pie_sizes, labels=pie_labels, colors=pie_colors,
                autopct='%1.1f%%', startangle=90,
                textprops={'fontsize': 11, 'fontweight': 'bold'})
    ax2.set_title('Severity Level Distribution', fontweight='bold')

    plt.tight_layout()

    if save_dir:
        save_dir.mkdir(parents=True, exist_ok=True)
        fig.savefig(save_dir / 'severity_distribution.png', dpi=150, bbox_inches='tight')

    plt.show()
    plt.close(fig)


if all_severity_results:
    plot_severity_distribution(all_severity_results, SEV_PLOTS_DIR)

---
## 16. Verify Output Files
List and verify all generated severity output files.

In [ ]:
def verify_outputs(output_dir: Path) -> None:
    """List and verify all generated severity outputs."""
    table = Table(
        title='📦 Severity Engine Outputs',
        show_header=True, header_style='bold cyan'
    )
    table.add_column('Directory', style='cyan', min_width=16)
    table.add_column('File', min_width=36)
    table.add_column('Size', justify='right')

    total_files = 0
    for subdir in sorted(output_dir.rglob('*')):
        if subdir.is_file():
            rel_dir = subdir.parent.relative_to(output_dir)
            size_kb = subdir.stat().st_size / 1024
            size_str = f'{size_kb:.1f} KB' if size_kb < 1024 else f'{size_kb/1024:.1f} MB'
            table.add_row(str(rel_dir), subdir.name, size_str)
            total_files += 1

    console.print(table)
    console.print(f'\n[bold]Total files generated: {total_files}[/bold]')


verify_outputs(SEVERITY_DIR)

---
## 17. Generate Backend-Ready Severity Module
Auto-generate a production-ready Python module for integration into the FastAPI backend.

In [ ]:
BACKEND_DIR = PROJECT_ROOT / 'backend_ready'
BACKEND_DIR.mkdir(parents=True, exist_ok=True)

severity_engine_py = textwrap.dedent('''\
"""
PlasticSense AI — Severity Engine Module
Auto-generated by Notebook 09: Severity Engine.

Usage:
    from severity_engine import SeverityEngine
    engine = SeverityEngine()
    result = engine.assess(detection_result, lat=19.12, lon=72.90)
"""
import math
import logging
import datetime
from typing import Dict, List, Any, Optional
from collections import Counter

logger = logging.getLogger("PlasticSense_AI")


# ──────────────────────────────────────────────────────────
# Configuration
# ──────────────────────────────────────────────────────────
HAZARD_WEIGHTS: Dict[str, int] = {
    "plastic_bottle": 1,
    "plastic_cap": 1,
    "plastic_bag": 2,
    "wrapper": 2,
    "food_container": 2,
    "styrofoam": 3,
    "multilayer_packaging": 4,
    "other_plastic": 2,
}

SEVERITY_LEVELS = [
    {"min": 0,  "max": 25,  "level": "Low",      "badge": "🟢", "color": "#2ecc71"},
    {"min": 26, "max": 50,  "level": "Medium",   "badge": "🟡", "color": "#f1c40f"},
    {"min": 51, "max": 75,  "level": "High",     "badge": "🟠", "color": "#e67e22"},
    {"min": 76, "max": 100, "level": "Critical", "badge": "🔴", "color": "#e74c3c"},
]

W_COUNT = 0.35
W_DENSITY = 0.30
W_HAZARD = 0.25
W_WATER = 0.10
MAX_EXPECTED_OBJECTS = 50
MAX_EXPECTED_DENSITY = 60.0
MAX_HAZARD_WEIGHT = max(HAZARD_WEIGHTS.values())


class SeverityEngine:
    """Environmental pollution severity assessment engine."""

    def __init__(self) -> None:
        logger.info("SeverityEngine initialized.")

    def assess(
        self,
        detection_result: Dict[str, Any],
        latitude: Optional[float] = None,
        longitude: Optional[float] = None,
        timestamp: Optional[str] = None,
    ) -> Dict[str, Any]:
        """Run full severity assessment on a detection result."""
        if detection_result.get("status") != "success":
            return {
                "status": "error",
                "severity": {"score": 0, "level": "N/A", "badge": "⚪"},
            }

        dets = detection_result["detections"]
        summary = detection_result["summary"]
        img_w = detection_result.get("image_width", 640)
        img_h = detection_result.get("image_height", 640)

        density = self.calculate_density(dets, img_w, img_h)
        hazard = self.calculate_hazard(dets)
        water = self.check_waterbody_proximity(latitude, longitude)

        count_score = min(summary["total_objects"] / MAX_EXPECTED_OBJECTS * 100, 100)
        severity_score = round(
            W_COUNT * count_score
            + W_DENSITY * density["density_score"]
            + W_HAZARD * hazard["hazard_score"]
            + W_WATER * water["waterbody_score"],
            2,
        )
        severity_score = max(0.0, min(100.0, severity_score))
        sev_class = self._classify(severity_score)

        return {
            "image_name": detection_result["image_name"],
            "timestamp": timestamp or datetime.datetime.now().isoformat(),
            "status": "success",
            "severity": {
                "score": severity_score,
                "level": sev_class["level"],
                "badge": sev_class["badge"],
            },
            "plastic_summary": {
                "total_objects": summary["total_objects"],
                "dominant_class": summary.get("dominant_type", "N/A"),
                "density": density["coverage_percentage"],
            },
            "hazard": {"score": hazard["hazard_score"]},
            "location": {
                "latitude": latitude,
                "longitude": longitude,
                "near_water": water["near_water"],
            },
        }

    @staticmethod
    def calculate_density(
        detections: List[Dict], img_w: int, img_h: int
    ) -> Dict[str, Any]:
        image_area = img_w * img_h
        if image_area == 0 or not detections:
            return {"coverage_percentage": 0.0, "density_score": 0.0}
        total_bbox = sum(d["bbox"][2] * d["bbox"][3] for d in detections)
        cov = (total_bbox / image_area) * 100
        return {
            "coverage_percentage": round(cov, 2),
            "density_score": round(min(cov / MAX_EXPECTED_DENSITY * 100, 100), 2),
        }

    @staticmethod
    def calculate_hazard(
        detections: List[Dict],
    ) -> Dict[str, Any]:
        if not detections:
            return {"hazard_score": 0.0, "raw_score": 0}
        counter = Counter(d["class"] for d in detections)
        raw = sum(cnt * HAZARD_WEIGHTS.get(cls, 2) for cls, cnt in counter.items())
        mx = len(detections) * MAX_HAZARD_WEIGHT
        return {
            "hazard_score": round(raw / mx * 100 if mx else 0, 2),
            "raw_score": raw,
        }

    @staticmethod
    def check_waterbody_proximity(
        lat: Optional[float] = None, lon: Optional[float] = None
    ) -> Dict[str, Any]:
        """Check proximity — replace with real GIS API in production."""
        if lat is None or lon is None:
            return {"near_water": False, "waterbody_score": 0.0}

        WATERBODIES = [
            {"name": "Yamuna River", "lat": 28.6139, "lon": 77.2090},
            {"name": "Ganges River", "lat": 25.3176, "lon": 82.9739},
            {"name": "Hussain Sagar", "lat": 17.4239, "lon": 78.4738},
            {"name": "Marina Beach", "lat": 13.0500, "lon": 80.2824},
            {"name": "Powai Lake", "lat": 19.1275, "lon": 72.9060},
        ]

        def _haversine(la1, lo1, la2, lo2):
            R = 6371.0
            la1r, la2r = math.radians(la1), math.radians(la2)
            dla = math.radians(la2 - la1)
            dlo = math.radians(lo2 - lo1)
            a = math.sin(dla / 2) ** 2 + math.cos(la1r) * math.cos(la2r) * math.sin(dlo / 2) ** 2
            return R * 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))

        dists = [(_haversine(lat, lon, w["lat"], w["lon"]), w) for w in WATERBODIES]
        dist, nearest = min(dists, key=lambda x: x[0])
        score = max(0, (1 - dist / 10) * 100) if dist < 10 else 0
        return {
            "near_water": dist <= 1.0,
            "waterbody_name": nearest["name"],
            "distance_km": round(dist, 2),
            "waterbody_score": round(score, 2),
        }

    @staticmethod
    def _classify(score: float) -> Dict[str, str]:
        for lvl in SEVERITY_LEVELS:
            if lvl["min"] <= score <= lvl["max"]:
                return lvl
        return SEVERITY_LEVELS[-1]
''')

severity_module_path = BACKEND_DIR / 'severity_engine.py'
with open(severity_module_path, 'w') as f:
    f.write(severity_engine_py)

console.print(f'[green]✔ Generated backend module: {severity_module_path}[/green]')
logger.info(f'Backend module generated: severity_engine.py')

# ── Verify backend module ──
be_table = Table(
    title='🏗️ Updated Backend Modules',
    show_header=True, header_style='bold blue'
)
be_table.add_column('File', style='cyan', min_width=24)
be_table.add_column('Size', justify='right')
be_table.add_column('Status', justify='center')

for fname in ['detector.py', 'utils.py', 'config.py', 'severity_engine.py', 'requirements.txt']:
    fpath = BACKEND_DIR / fname
    if fpath.exists():
        size_kb = fpath.stat().st_size / 1024
        be_table.add_row(fname, f'{size_kb:.1f} KB', '[bold green]✔[/bold green]')
    else:
        be_table.add_row(fname, 'N/A', '[yellow]— (from NB08)[/yellow]')

console.print(be_table)

---
## 18. ✅ Final Severity Engine Summary
Display a comprehensive summary of the entire severity assessment pipeline.

In [ ]:
def display_final_summary(
    severity_results: List[Dict[str, Any]],
    severity_dir: Path,
    backend_dir: Path
) -> None:
    """Display the final severity engine summary."""
    successful = [r for r in severity_results if r.get('status') == 'success']

    total_objects = sum(r['plastic_summary']['total_objects'] for r in successful) if successful else 0
    avg_density = np.mean([r['plastic_summary']['density'] for r in successful]) if successful else 0
    avg_hazard = np.mean([r['hazard']['score'] for r in successful]) if successful else 0
    avg_severity = np.mean([r['severity']['score'] for r in successful]) if successful else 0

    # Most common level
    if successful:
        levels = Counter(r['severity']['level'] for r in successful)
        common_level = levels.most_common(1)[0]
        level_str = f'{common_level[0]} ({common_level[1]} images)'
    else:
        level_str = 'N/A'

    # Count generated files
    json_count = len(list(SEV_JSON_DIR.glob('*.json'))) if SEV_JSON_DIR.exists() else 0
    csv_count = len(list(SEV_CSV_DIR.glob('*.csv'))) if SEV_CSV_DIR.exists() else 0
    plot_count = len(list(SEV_PLOTS_DIR.glob('*.png'))) if SEV_PLOTS_DIR.exists() else 0
    backend_exists = (backend_dir / 'severity_engine.py').exists()

    summary_text = (
        f'[bold green]✔ Plastic Objects Detected[/bold green]    {total_objects}\n'
        f'[bold green]✔ Average Density[/bold green]             {avg_density:.2f}%\n'
        f'[bold green]✔ Average Hazard Score[/bold green]        {avg_hazard:.2f}\n'
        f'[bold green]✔ Average Severity Score[/bold green]      {avg_severity:.2f}\n'
        f'[bold green]✔ Most Common Severity[/bold green]        {level_str}\n'
        f'[bold green]✔ JSON Files Generated[/bold green]        {json_count}\n'
        f'[bold green]✔ CSV Files Generated[/bold green]         {csv_count}\n'
        f'[bold green]✔ Plots Generated[/bold green]             {plot_count}\n'
        f'[bold green]✔ Backend Module[/bold green]              {"✔ severity_engine.py" if backend_exists else "✖"}\n'
        f'[bold green]✔ Backend Ready[/bold green]\n'
        f'\n'
        f'[bold cyan]─── Output Locations ───[/bold cyan]\n'
        f'  📁 JSON Reports:     {SEV_JSON_DIR}\n'
        f'  📁 CSV Reports:      {SEV_CSV_DIR}\n'
        f'  📁 Plots:            {SEV_PLOTS_DIR}\n'
        f'  📁 Summary Reports:  {SEV_REPORTS_DIR}\n'
        f'  📁 Backend Module:   {backend_dir / "severity_engine.py"}\n'
        f'\n'
        f'[bold red]Next Notebook:[/bold red]  10_Disposal_Recommendation_Engine.ipynb\n'
        f'The outputs of this notebook will be integrated into the FastAPI\n'
        f'backend to automatically determine the urgency of cleanup operations.'
    )

    console.print(Panel.fit(
        summary_text,
        title='🏁 PlasticSense AI — Severity Engine Complete',
        border_style='bold green'
    ))


display_final_summary(all_severity_results, SEVERITY_DIR, BACKEND_DIR)
logger.info('=== SEVERITY ENGINE PIPELINE COMPLETE ===')